# 3D Ising-like porous media workflow

This notebook includes:

- synthetic porous morphology generation with an Ising-like lattice model,
- segmented micro-CT `.raw` import,
- porosity, connectivity, tortuosity proxy,
- 1D correlation and full 3D radial autocorrelation,
- specific surface area,
- comparison between synthetic and imported CT,
- CFD/OpenFOAM-oriented export.

In [ ]:

from __future__ import annotations

import json
import math
import os
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

In [ ]:

@dataclass
class SimulationConfig:
    shape: Tuple[int, int, int] = (32, 32, 32)
    porosity: float = 0.35
    temperature: float = 0.9
    Jx: float = 1.0
    Jy: float = 1.0
    Jz: float = 1.0
    sweeps: int = 15
    seed: int = 7
    report_every: int = 3

cfg = SimulationConfig(
    shape=(32, 32, 32),
    porosity=0.35,
    temperature=0.9,
    Jx=1.0,
    Jy=1.0,
    Jz=1.0,
    sweeps=15,
    seed=7,
    report_every=3,
)

print(cfg)

In [ ]:

def validate_config(cfg: SimulationConfig) -> None:
    if len(cfg.shape) != 3:
        raise ValueError("shape must have three entries: (nx, ny, nz)")
    if not (0.0 < cfg.porosity < 1.0):
        raise ValueError("porosity must be between 0 and 1")
    if cfg.temperature <= 0.0:
        raise ValueError("temperature must be positive")
    if cfg.sweeps < 1:
        raise ValueError("sweeps must be >= 1")


def create_initial_lattice(shape: Tuple[int, int, int], porosity: float, rng: np.random.Generator) -> np.ndarray:
    n_sites = int(np.prod(shape))
    n_pores = int(round(porosity * n_sites))
    flat = np.ones(n_sites, dtype=np.int8)
    flat[:n_pores] = 0
    rng.shuffle(flat)
    return flat.reshape(shape)


def spin_value(lattice: np.ndarray, idx: Tuple[int, int, int]) -> int:
    return 1 if lattice[idx] == 1 else -1


def wrapped_neighbor(idx: Tuple[int, int, int], axis: int, delta: int, shape: Tuple[int, int, int]) -> Tuple[int, int, int]:
    coords = list(idx)
    coords[axis] = (coords[axis] + delta) % shape[axis]
    return tuple(coords)


def positive_direction_bonds(shape: Tuple[int, int, int], idx: Tuple[int, int, int]):
    return [
        (idx, wrapped_neighbor(idx, 0, 1, shape), 0.0),
        (idx, wrapped_neighbor(idx, 1, 1, shape), 0.0),
        (idx, wrapped_neighbor(idx, 2, 1, shape), 0.0),
    ]


def bond_axis(a: Tuple[int, int, int], b: Tuple[int, int, int], shape: Tuple[int, int, int]) -> int:
    for axis in range(3):
        if wrapped_neighbor(a, axis, +1, shape) == b or wrapped_neighbor(a, axis, -1, shape) == b:
            return axis
    raise ValueError("Sites are not nearest neighbors under periodic boundaries")


def coupling_for_axis(axis: int, cfg: SimulationConfig) -> float:
    if axis == 0:
        return cfg.Jx
    if axis == 1:
        return cfg.Jy
    if axis == 2:
        return cfg.Jz
    raise ValueError("axis must be 0, 1, or 2")


def bond_energy(lattice: np.ndarray, a: Tuple[int, int, int], b: Tuple[int, int, int], cfg: SimulationConfig) -> float:
    axis = bond_axis(a, b, lattice.shape)
    J = coupling_for_axis(axis, cfg)
    sa = spin_value(lattice, a)
    sb = spin_value(lattice, b)
    return -J * sa * sb


def total_energy(lattice: np.ndarray, cfg: SimulationConfig) -> float:
    e = 0.0
    nx, ny, nz = lattice.shape
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                idx = (i, j, k)
                for a, b, _ in positive_direction_bonds(lattice.shape, idx):
                    e += bond_energy(lattice, a, b, cfg)
    return e


def incident_bonds(idx: Tuple[int, int, int], shape: Tuple[int, int, int]):
    bonds = []
    for axis in range(3):
        bonds.append((idx, wrapped_neighbor(idx, axis, +1, shape)))
        bonds.append((idx, wrapped_neighbor(idx, axis, -1, shape)))
    return bonds


def unique_bond_key(a: Tuple[int, int, int], b: Tuple[int, int, int]):
    return (a, b) if a <= b else (b, a)


def delta_energy_pair_exchange(lattice: np.ndarray, a: Tuple[int, int, int], b: Tuple[int, int, int], cfg: SimulationConfig) -> float:
    touched = {}
    for x, y in incident_bonds(a, lattice.shape) + incident_bonds(b, lattice.shape):
        touched[unique_bond_key(x, y)] = (x, y)
    e_before = sum(bond_energy(lattice, x, y, cfg) for x, y in touched.values())
    lattice[a], lattice[b] = lattice[b], lattice[a]
    e_after = sum(bond_energy(lattice, x, y, cfg) for x, y in touched.values())
    lattice[a], lattice[b] = lattice[b], lattice[a]
    return e_after - e_before


def random_neighbor(idx: Tuple[int, int, int], shape: Tuple[int, int, int], rng: np.random.Generator) -> Tuple[int, int, int]:
    axis = int(rng.integers(0, 3))
    delta = 1 if rng.random() < 0.5 else -1
    return wrapped_neighbor(idx, axis, delta, shape)


def metropolis_accept(delta_e: float, temperature: float, rng: np.random.Generator) -> bool:
    if delta_e <= 0.0:
        return True
    return rng.random() < math.exp(-delta_e / temperature)


def monte_carlo_sweep(lattice: np.ndarray, cfg: SimulationConfig, rng: np.random.Generator) -> int:
    nx, ny, nz = lattice.shape
    n_attempts = nx * ny * nz
    accepted = 0
    for _ in range(n_attempts):
        a = (
            int(rng.integers(0, nx)),
            int(rng.integers(0, ny)),
            int(rng.integers(0, nz)),
        )
        b = random_neighbor(a, lattice.shape, rng)
        if lattice[a] == lattice[b]:
            continue
        dE = delta_energy_pair_exchange(lattice, a, b, cfg)
        if metropolis_accept(dE, cfg.temperature, rng):
            lattice[a], lattice[b] = lattice[b], lattice[a]
            accepted += 1
    return accepted


def measured_porosity(lattice: np.ndarray) -> float:
    return float(np.mean(lattice == 0))


def interface_fraction(lattice: np.ndarray) -> float:
    unlike = 0
    total = 0
    nx, ny, nz = lattice.shape
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                a = (i, j, k)
                for b in [
                    wrapped_neighbor(a, 0, +1, lattice.shape),
                    wrapped_neighbor(a, 1, +1, lattice.shape),
                    wrapped_neighbor(a, 2, +1, lattice.shape),
                ]:
                    unlike += int(lattice[a] != lattice[b])
                    total += 1
    return unlike / total


def one_dimensional_correlation(lattice: np.ndarray, axis: int = 0, max_lag: Optional[int] = None):
    spins = np.where(lattice == 1, 1.0, -1.0)
    n = lattice.shape[axis]
    if max_lag is None:
        max_lag = n // 2
    lags = np.arange(max_lag + 1)
    corr = np.zeros_like(lags, dtype=float)
    for lag in lags:
        shifted = np.roll(spins, shift=-lag, axis=axis)
        corr[lag] = np.mean(spins * shifted)
    return lags, corr


def normalize_correlation(corr: np.ndarray) -> np.ndarray:
    if len(corr) == 0:
        return corr.copy()
    c0 = corr[0]
    if np.isclose(c0, 0.0):
        return corr.copy()
    return corr / c0


def first_zero_crossing(lags: np.ndarray, corr: np.ndarray) -> Optional[float]:
    for i in range(1, len(corr)):
        if corr[i] == 0.0:
            return float(lags[i])
        if corr[i - 1] > 0.0 and corr[i] < 0.0:
            x1, x2 = float(lags[i - 1]), float(lags[i])
            y1, y2 = float(corr[i - 1]), float(corr[i])
            return x1 - y1 * (x2 - x1) / (y2 - y1)
    return None


def decay_length_1_over_e(lags: np.ndarray, corr: np.ndarray) -> Optional[float]:
    corr_n = normalize_correlation(corr)
    target = math.exp(-1.0)
    for i in range(1, len(corr_n)):
        if corr_n[i] == target:
            return float(lags[i])
        if corr_n[i - 1] >= target and corr_n[i] < target:
            x1, x2 = float(lags[i - 1]), float(lags[i])
            y1, y2 = float(corr_n[i - 1]), float(corr_n[i])
            return x1 + (target - y1) * (x2 - x1) / (y2 - y1)
    return None


def correlation_length_summary(lattice: np.ndarray) -> Dict[str, Optional[float]]:
    summary: Dict[str, Optional[float]] = {}
    axis_names = {0: "x", 1: "y", 2: "z"}
    for axis in range(3):
        lags, corr = one_dimensional_correlation(lattice, axis=axis)
        summary[f"zero_crossing_{axis_names[axis]}"] = first_zero_crossing(lags, corr)
        summary[f"decay_1_over_e_{axis_names[axis]}"] = decay_length_1_over_e(lags, corr)
    zero_vals = [summary[f"zero_crossing_{ax}"] for ax in ["x", "y", "z"] if summary[f"zero_crossing_{ax}"] is not None]
    decay_vals = [summary[f"decay_1_over_e_{ax}"] for ax in ["x", "y", "z"] if summary[f"decay_1_over_e_{ax}"] is not None]
    summary["mean_zero_crossing"] = float(np.mean(zero_vals)) if zero_vals else None
    summary["mean_decay_1_over_e"] = float(np.mean(decay_vals)) if decay_vals else None
    return summary


def run_simulation(cfg: SimulationConfig):
    validate_config(cfg)
    rng = np.random.default_rng(cfg.seed)
    lattice = create_initial_lattice(cfg.shape, cfg.porosity, rng)
    history = {"sweep": [], "energy": [], "acceptance_rate": [], "interface_fraction": [], "porosity": []}
    n_sites = int(np.prod(cfg.shape))
    print("Starting simulation...")
    for sweep in range(1, cfg.sweeps + 1):
        accepted = monte_carlo_sweep(lattice, cfg, rng)
        history["sweep"].append(sweep)
        history["energy"].append(total_energy(lattice, cfg) / n_sites)
        history["acceptance_rate"].append(accepted / n_sites)
        history["interface_fraction"].append(interface_fraction(lattice))
        history["porosity"].append(measured_porosity(lattice))
        if sweep == 1 or sweep % cfg.report_every == 0 or sweep == cfg.sweeps:
            print(
                f"Sweep {sweep:4d} | E/N = {history['energy'][-1]: .4f} | "
                f"acc = {history['acceptance_rate'][-1]: .4f} | interface = {history['interface_fraction'][-1]: .4f}"
            )
    return lattice, history

In [ ]:

def porous_cmap():
    return ListedColormap(["royalblue", "firebrick"])


def plot_history(history):
    sweeps = history["sweep"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(sweeps, history["energy"], marker="o", ms=3)
    axes[0].set_title("Energy per site")
    axes[0].set_xlabel("Sweep")
    axes[0].set_ylabel("E / N")
    axes[1].plot(sweeps, history["acceptance_rate"], marker="o", ms=3)
    axes[1].set_title("Acceptance rate")
    axes[1].set_xlabel("Sweep")
    axes[1].set_ylabel("Accepted moves / N")
    axes[2].plot(sweeps, history["interface_fraction"], marker="o", ms=3, label="Interface fraction")
    axes[2].plot(sweeps, history["porosity"], marker="s", ms=3, label="Porosity")
    axes[2].set_title("Morphology diagnostics")
    axes[2].set_xlabel("Sweep")
    axes[2].legend()
    plt.tight_layout()
    plt.show()


def plot_slices(lattice, title="Morphology slices"):
    nx, ny, nz = lattice.shape
    cx, cy, cz = nx // 2, ny // 2, nz // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    cmap = porous_cmap()
    axes[0].imshow(lattice[:, :, cz].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
    axes[0].set_title(f"XY slice at z={cz}")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")
    axes[1].imshow(lattice[:, cy, :].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
    axes[1].set_title(f"XZ slice at y={cy}")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("z")
    axes[2].imshow(lattice[cx, :, :].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
    axes[2].set_title(f"YZ slice at x={cx}")
    axes[2].set_xlabel("y")
    axes[2].set_ylabel("z")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_correlation(lags, corr, axis_name="x"):
    plt.figure(figsize=(6, 4))
    plt.plot(lags, corr, marker="o", ms=4)
    plt.axhline(0.0, ls="--")
    plt.xlabel(f"Lag along {axis_name}")
    plt.ylabel("Correlation")
    plt.title(f"Two-point correlation along {axis_name}")
    plt.tight_layout()
    plt.show()


def plot_correlations_xyz(lattice):
    plt.figure(figsize=(7, 4.5))
    for axis, axis_name in zip([0, 1, 2], ["x", "y", "z"]):
        lags, corr = one_dimensional_correlation(lattice, axis=axis)
        plt.plot(lags, normalize_correlation(corr), marker="o", ms=3, label=f"{axis_name}-axis")
    plt.axhline(0.0, ls="--")
    plt.axhline(math.exp(-1.0), ls=":")
    plt.xlabel("Lag")
    plt.ylabel("Normalized correlation")
    plt.title("Normalized two-point correlations along x, y, and z")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:

def neighbors_open(idx: Tuple[int, int, int], shape: Tuple[int, int, int]):
    i, j, k = idx
    nx, ny, nz = shape
    nbrs = []
    if i > 0:
        nbrs.append((i - 1, j, k))
    if i < nx - 1:
        nbrs.append((i + 1, j, k))
    if j > 0:
        nbrs.append((i, j - 1, k))
    if j < ny - 1:
        nbrs.append((i, j + 1, k))
    if k > 0:
        nbrs.append((i, j, k - 1))
    if k < nz - 1:
        nbrs.append((i, j, k + 1))
    return nbrs


def label_phase_clusters(lattice: np.ndarray, phase: int = 0):
    labels = np.zeros(lattice.shape, dtype=np.int32)
    current_label = 0
    sizes = []
    phase_mask = lattice == phase
    for start in zip(*np.where(phase_mask)):
        if labels[start] != 0:
            continue
        current_label += 1
        q = deque([start])
        labels[start] = current_label
        cluster_size = 0
        while q:
            node = q.popleft()
            cluster_size += 1
            for nbr in neighbors_open(node, lattice.shape):
                if phase_mask[nbr] and labels[nbr] == 0:
                    labels[nbr] = current_label
                    q.append(nbr)
        sizes.append(cluster_size)
    return labels, sizes


def largest_cluster_mask(labels: np.ndarray, sizes: List[int]) -> np.ndarray:
    if not sizes:
        return np.zeros_like(labels, dtype=bool)
    largest_label = int(np.argmax(sizes)) + 1
    return labels == largest_label


def cluster_touches_both_faces(mask: np.ndarray, axis: int) -> bool:
    if axis == 0:
        return bool(np.any(mask[0, :, :]) and np.any(mask[-1, :, :]))
    if axis == 1:
        return bool(np.any(mask[:, 0, :]) and np.any(mask[:, -1, :]))
    if axis == 2:
        return bool(np.any(mask[:, :, 0]) and np.any(mask[:, :, -1]))
    raise ValueError("axis must be 0, 1, or 2")


def percolates_along_axis(lattice: np.ndarray, axis: int, phase: int = 0) -> bool:
    labels, sizes = label_phase_clusters(lattice, phase=phase)
    if not sizes:
        return False
    for label in range(1, len(sizes) + 1):
        if cluster_touches_both_faces(labels == label, axis):
            return True
    return False


def connected_pore_statistics(lattice: np.ndarray):
    labels, sizes = label_phase_clusters(lattice, phase=0)
    total_pores = int(np.sum(lattice == 0))
    if total_pores == 0 or not sizes:
        return {
            "total_pores": total_pores,
            "num_pore_clusters": 0,
            "largest_pore_cluster": 0,
            "largest_cluster_fraction_of_pores": 0.0,
            "percolates_x": False,
            "percolates_y": False,
            "percolates_z": False,
        }
    largest = int(np.max(sizes))
    return {
        "total_pores": total_pores,
        "num_pore_clusters": int(len(sizes)),
        "largest_pore_cluster": largest,
        "largest_cluster_fraction_of_pores": largest / total_pores,
        "percolates_x": percolates_along_axis(lattice, 0, 0),
        "percolates_y": percolates_along_axis(lattice, 1, 0),
        "percolates_z": percolates_along_axis(lattice, 2, 0),
    }


def plot_boolean_slices(mask, title="Boolean mask slices"):
    nx, ny, nz = mask.shape
    cx, cy, cz = nx // 2, ny // 2, nz // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    cmap = ListedColormap(["white", "navy"])
    axes[0].imshow(mask[:, :, cz].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
    axes[0].set_title(f"XY at z={cz}")
    axes[1].imshow(mask[:, cy, :].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
    axes[1].set_title(f"XZ at y={cy}")
    axes[2].imshow(mask[cx, :, :].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
    axes[2].set_title(f"YZ at x={cx}")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def porosity_profile(lattice: np.ndarray, axis: int = 2) -> np.ndarray:
    if axis == 0:
        return np.mean(lattice == 0, axis=(1, 2))
    if axis == 1:
        return np.mean(lattice == 0, axis=(0, 2))
    if axis == 2:
        return np.mean(lattice == 0, axis=(0, 1))
    raise ValueError("axis must be 0, 1, or 2")


def interface_profile(lattice: np.ndarray, axis: int = 2) -> np.ndarray:
    if axis == 0:
        slices = [lattice[i, :, :] for i in range(lattice.shape[0])]
    elif axis == 1:
        slices = [lattice[:, j, :] for j in range(lattice.shape[1])]
    elif axis == 2:
        slices = [lattice[:, :, k] for k in range(lattice.shape[2])]
    else:
        raise ValueError("axis must be 0, 1, or 2")
    values = []
    for sl in slices:
        ny, nx = sl.shape
        unlike = 0
        total = 0
        for i in range(ny):
            for j in range(nx):
                if i + 1 < ny:
                    unlike += int(sl[i, j] != sl[i + 1, j])
                    total += 1
                if j + 1 < nx:
                    unlike += int(sl[i, j] != sl[i, j + 1])
                    total += 1
        values.append(unlike / total if total > 0 else 0.0)
    return np.array(values, dtype=float)

In [ ]:

def label_clusters_2d(binary_slice: np.ndarray, phase: int = 0):
    ny, nx = binary_slice.shape
    labels = np.zeros((ny, nx), dtype=np.int32)
    current_label = 0
    sizes = []

    def neighbors2d(i: int, j: int):
        nbrs = []
        if i > 0:
            nbrs.append((i - 1, j))
        if i < ny - 1:
            nbrs.append((i + 1, j))
        if j > 0:
            nbrs.append((i, j - 1))
        if j < nx - 1:
            nbrs.append((i, j + 1))
        return nbrs

    mask = binary_slice == phase
    for start in zip(*np.where(mask)):
        if labels[start] != 0:
            continue
        current_label += 1
        q = deque([start])
        labels[start] = current_label
        cluster_size = 0
        while q:
            i, j = q.popleft()
            cluster_size += 1
            for ni, nj in neighbors2d(i, j):
                if mask[ni, nj] and labels[ni, nj] == 0:
                    labels[ni, nj] = current_label
                    q.append((ni, nj))
        sizes.append(cluster_size)
    return labels, sizes


def slice_connectivity_profile(lattice: np.ndarray, axis: int = 2):
    if axis == 0:
        slices = [lattice[i, :, :] for i in range(lattice.shape[0])]
    elif axis == 1:
        slices = [lattice[:, j, :] for j in range(lattice.shape[1])]
    elif axis == 2:
        slices = [lattice[:, :, k] for k in range(lattice.shape[2])]
    else:
        raise ValueError("axis must be 0, 1, or 2")
    num_clusters = []
    largest_sizes = []
    largest_fractions = []
    spans_h = []
    spans_v = []
    for sl in slices:
        labels, sizes = label_clusters_2d(sl, phase=0)
        total_pores = int(np.sum(sl == 0))
        num_clusters.append(len(sizes))
        if total_pores == 0 or not sizes:
            largest_sizes.append(0)
            largest_fractions.append(0.0)
            spans_h.append(False)
            spans_v.append(False)
            continue
        largest_label = int(np.argmax(sizes)) + 1
        largest_mask = labels == largest_label
        largest_sizes.append(int(np.max(sizes)))
        largest_fractions.append(float(np.max(sizes)) / total_pores)
        spans_h.append(bool(np.any(largest_mask[:, 0]) and np.any(largest_mask[:, -1])))
        spans_v.append(bool(np.any(largest_mask[0, :]) and np.any(largest_mask[-1, :])))
    return {
        "num_clusters": np.array(num_clusters, dtype=int),
        "largest_cluster_size": np.array(largest_sizes, dtype=int),
        "largest_cluster_fraction": np.array(largest_fractions, dtype=float),
        "spans_horizontal": np.array(spans_h, dtype=bool),
        "spans_vertical": np.array(spans_v, dtype=bool),
    }


def plot_depth_profiles(lattice, axis=2):
    phi = porosity_profile(lattice, axis=axis)
    inter = interface_profile(lattice, axis=axis)
    depth = np.arange(len(phi))
    axis_name = {0: "x", 1: "y", 2: "z"}[axis]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(depth, phi, marker="o", ms=3)
    axes[0].set_title(f"Porosity profile along {axis_name}")
    axes[0].set_xlabel(f"Slice index along {axis_name}")
    axes[0].set_ylabel("Porosity")
    axes[1].plot(depth, inter, marker="o", ms=3)
    axes[1].set_title(f"In-slice interface profile along {axis_name}")
    axes[1].set_xlabel(f"Slice index along {axis_name}")
    axes[1].set_ylabel("Interface fraction")
    plt.tight_layout()
    plt.show()


def plot_slice_connectivity_profiles(lattice, axis=2):
    stats = slice_connectivity_profile(lattice, axis=axis)
    depth = np.arange(len(stats["num_clusters"]))
    axis_name = {0: "x", 1: "y", 2: "z"}[axis]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(depth, stats["num_clusters"], marker="o", ms=3)
    axes[0].set_title(f"Number of pore clusters per {axis_name}-slice")
    axes[1].plot(depth, stats["largest_cluster_fraction"], marker="o", ms=3)
    axes[1].set_title(f"Largest pore-cluster fraction per {axis_name}-slice")
    axes[2].plot(depth, stats["spans_horizontal"].astype(int), marker="o", ms=3, label="Horizontal span")
    axes[2].plot(depth, stats["spans_vertical"].astype(int), marker="s", ms=3, label="Vertical span")
    axes[2].set_title(f"Lateral spanning inside each {axis_name}-slice")
    axes[2].legend()
    plt.tight_layout()
    plt.show()

In [ ]:

def axis_face_pores(lattice: np.ndarray, axis: int, side: int, phase: int = 0):
    if axis not in (0, 1, 2):
        raise ValueError("axis must be 0, 1, or 2")
    if side not in (0, 1):
        raise ValueError("side must be 0 or 1")
    shape = lattice.shape
    index = 0 if side == 0 else shape[axis] - 1
    points = []
    if axis == 0:
        for j in range(shape[1]):
            for k in range(shape[2]):
                if lattice[index, j, k] == phase:
                    points.append((index, j, k))
    elif axis == 1:
        for i in range(shape[0]):
            for k in range(shape[2]):
                if lattice[i, index, k] == phase:
                    points.append((i, index, k))
    else:
        for i in range(shape[0]):
            for j in range(shape[1]):
                if lattice[i, j, index] == phase:
                    points.append((i, j, index))
    return points


def shortest_path_distances_from_face(lattice: np.ndarray, axis: int, phase: int = 0) -> np.ndarray:
    dist = -np.ones(lattice.shape, dtype=np.int32)
    q = deque()
    for src in axis_face_pores(lattice, axis=axis, side=0, phase=phase):
        dist[src] = 0
        q.append(src)
    while q:
        node = q.popleft()
        for nbr in neighbors_open(node, lattice.shape):
            if lattice[nbr] != phase:
                continue
            if dist[nbr] != -1:
                continue
            dist[nbr] = dist[node] + 1
            q.append(nbr)
    return dist


def shortest_path_to_opposite_face(lattice: np.ndarray, axis: int, phase: int = 0) -> Optional[int]:
    dist = shortest_path_distances_from_face(lattice, axis=axis, phase=phase)
    target_distances = [int(dist[t]) for t in axis_face_pores(lattice, axis=axis, side=1, phase=phase) if dist[t] >= 0]
    return min(target_distances) if target_distances else None


def geodesic_tortuosity(lattice: np.ndarray, axis: int, phase: int = 0) -> Optional[float]:
    shortest = shortest_path_to_opposite_face(lattice, axis=axis, phase=phase)
    if shortest is None:
        return None
    straight = lattice.shape[axis] - 1
    if straight <= 0:
        return None
    return float(shortest) / float(straight)


def accessible_fraction_from_inlet(lattice: np.ndarray, axis: int, phase: int = 0) -> float:
    total_phase = int(np.sum(lattice == phase))
    if total_phase == 0:
        return 0.0
    dist = shortest_path_distances_from_face(lattice, axis=axis, phase=phase)
    reachable = int(np.sum(dist >= 0))
    return reachable / total_phase


def breakthrough_fraction_profile(lattice, axis=2, phase=0):
    dist = shortest_path_distances_from_face(lattice, axis=axis, phase=phase)
    reachable = (dist >= 0) & (lattice == phase)
    if axis == 0:
        denom = np.sum(lattice == phase, axis=(1, 2))
        num = np.sum(reachable, axis=(1, 2))
    elif axis == 1:
        denom = np.sum(lattice == phase, axis=(0, 2))
        num = np.sum(reachable, axis=(0, 2))
    elif axis == 2:
        denom = np.sum(lattice == phase, axis=(0, 1))
        num = np.sum(reachable, axis=(0, 1))
    else:
        raise ValueError("axis must be 0, 1, or 2")
    profile = np.zeros_like(num, dtype=float)
    mask = denom > 0
    profile[mask] = num[mask] / denom[mask]
    return profile


def tortuosity_summary(lattice: np.ndarray):
    summary = {}
    for axis, name in zip([0, 1, 2], ["x", "y", "z"]):
        tau = geodesic_tortuosity(lattice, axis=axis, phase=0)
        summary[f"percolates_{name}"] = tau is not None
        summary[f"geodesic_tortuosity_{name}"] = tau
        summary[f"accessible_fraction_from_{name}_inlet"] = accessible_fraction_from_inlet(lattice, axis=axis, phase=0)
    return summary


def plot_breakthrough_profile(lattice, axis=2):
    prof = breakthrough_fraction_profile(lattice, axis=axis, phase=0)
    depth = np.arange(len(prof))
    axis_name = {0: "x", 1: "y", 2: "z"}[axis]
    plt.figure(figsize=(6.5, 4))
    plt.plot(depth, prof, marker="o", ms=3)
    plt.xlabel(f"Slice index along {axis_name}")
    plt.ylabel("Reachable pore fraction")
    plt.title(f"Breakthrough profile from the {axis_name}-inlet face")
    plt.tight_layout()
    plt.show()


def plot_distance_field_slices(lattice, axis=2):
    dist = shortest_path_distances_from_face(lattice, axis=axis, phase=0)
    nx, ny, nz = dist.shape
    cx, cy, cz = nx // 2, ny // 2, nz // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    im0 = axes[0].imshow(dist[:, :, cz].T, origin="lower")
    axes[0].set_title(f"Distance field: XY at z={cz}")
    im1 = axes[1].imshow(dist[:, cy, :].T, origin="lower")
    axes[1].set_title(f"Distance field: XZ at y={cy}")
    im2 = axes[2].imshow(dist[cx, :, :].T, origin="lower")
    axes[2].set_title(f"Distance field: YZ at x={cx}")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    fig.suptitle("Shortest pore-path distance from inlet face", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:

def _normalize_spacing(spacing: Optional[Union[float, Tuple[float, float, float]]]) -> Tuple[float, float, float]:
    if spacing is None:
        return (1.0, 1.0, 1.0)
    if isinstance(spacing, (int, float)):
        s = float(spacing)
        return (s, s, s)
    if len(spacing) != 3:
        raise ValueError("spacing must be a scalar or tuple of length 3")
    return (float(spacing[0]), float(spacing[1]), float(spacing[2]))


def spin_field(volume: np.ndarray) -> np.ndarray:
    return np.where(volume == 1, 1.0, -1.0)


def autocorrelation_3d_fft(volume: np.ndarray) -> np.ndarray:
    spins = spin_field(volume)
    f = np.fft.fftn(spins)
    return np.fft.ifftn(f * np.conjugate(f)).real / spins.size


def radial_distance_grid(shape: Tuple[int, int, int], spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> np.ndarray:
    sx, sy, sz = _normalize_spacing(spacing)
    dx = np.fft.fftfreq(shape[0]) * shape[0] * sx
    dy = np.fft.fftfreq(shape[1]) * shape[1] * sy
    dz = np.fft.fftfreq(shape[2]) * shape[2] * sz
    gx, gy, gz = np.meshgrid(dx, dy, dz, indexing="ij")
    return np.sqrt(gx**2 + gy**2 + gz**2)


def radial_average_3d(field: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None, bin_width: float = 1.0):
    if bin_width <= 0.0:
        raise ValueError("bin_width must be positive")
    r = radial_distance_grid(field.shape, spacing=spacing)
    shell_index = np.rint(r.ravel() / bin_width).astype(int)
    sums = np.bincount(shell_index, weights=field.ravel())
    counts = np.bincount(shell_index)
    valid = counts > 0
    radii = np.arange(len(sums), dtype=float) * bin_width
    values = np.zeros_like(radii, dtype=float)
    values[valid] = sums[valid] / counts[valid]
    return radii[valid], values[valid], counts[valid]


def full_3d_radial_correlation(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None, bin_width: float = 1.0):
    corr3d = autocorrelation_3d_fft(volume)
    return radial_average_3d(corr3d, spacing=spacing, bin_width=bin_width)


def radial_correlation_summary(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None, bin_width: float = 1.0):
    r, c_r, _ = full_3d_radial_correlation(volume, spacing=spacing, bin_width=bin_width)
    c_norm = normalize_correlation(c_r)
    return {
        "radial_zero_crossing": first_zero_crossing(r, c_norm),
        "radial_decay_1_over_e": decay_length_1_over_e(r, c_norm),
    }


def plot_full_3d_radial_correlation(volume, spacing=None, bin_width=1.0, title="Full 3D radial autocorrelation", label=None):
    r, c_r, counts = full_3d_radial_correlation(volume, spacing=spacing, bin_width=bin_width)
    c_norm = normalize_correlation(c_r)
    plt.figure(figsize=(6.5, 4.2))
    plt.plot(r, c_norm, marker="o", ms=3, label=label if label else None)
    plt.axhline(0.0, ls="--")
    plt.axhline(math.exp(-1.0), ls=":")
    plt.xlabel("Radial distance")
    plt.ylabel("Normalized radial autocorrelation")
    plt.title(title)
    if label:
        plt.legend()
    plt.tight_layout()
    plt.show()
    return r, c_norm, counts


def compare_full_3d_radial_correlations(volume_a, volume_b, spacing_a=None, spacing_b=None, bin_width_a=1.0, bin_width_b=1.0, label_a="A", label_b="B", title="Comparison of full 3D radial autocorrelations"):
    r_a, c_a, _ = full_3d_radial_correlation(volume_a, spacing=spacing_a, bin_width=bin_width_a)
    r_b, c_b, _ = full_3d_radial_correlation(volume_b, spacing=spacing_b, bin_width=bin_width_b)
    c_a = normalize_correlation(c_a)
    c_b = normalize_correlation(c_b)
    plt.figure(figsize=(7, 4.5))
    plt.plot(r_a, c_a, marker="o", ms=3, label=label_a)
    plt.plot(r_b, c_b, marker="s", ms=3, label=label_b)
    plt.axhline(0.0, ls="--")
    plt.axhline(math.exp(-1.0), ls=":")
    plt.xlabel("Radial distance")
    plt.ylabel("Normalized radial autocorrelation")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:

def voxel_spacing_tuple(spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> Tuple[float, float, float]:
    return _normalize_spacing(spacing)


def interface_area_counts(volume: np.ndarray):
    cx = int(np.sum(volume[:-1, :, :] != volume[1:, :, :]))
    cy = int(np.sum(volume[:, :-1, :] != volume[:, 1:, :]))
    cz = int(np.sum(volume[:, :, :-1] != volume[:, :, 1:]))
    return {"faces_x": cx, "faces_y": cy, "faces_z": cz, "faces_total": cx + cy + cz}


def interfacial_area(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> float:
    dx, dy, dz = voxel_spacing_tuple(spacing)
    counts = interface_area_counts(volume)
    area = counts["faces_x"] * dy * dz + counts["faces_y"] * dx * dz + counts["faces_z"] * dx * dy
    return float(area)


def bulk_volume(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> float:
    dx, dy, dz = voxel_spacing_tuple(spacing)
    return float(volume.size * dx * dy * dz)


def pore_volume(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> float:
    dx, dy, dz = voxel_spacing_tuple(spacing)
    return float(np.sum(volume == 0) * dx * dy * dz)


def specific_surface_area_bulk(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> float:
    area = interfacial_area(volume, spacing=spacing)
    vbulk = bulk_volume(volume, spacing=spacing)
    return 0.0 if vbulk <= 0.0 else float(area / vbulk)


def specific_surface_area_pore(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> float:
    area = interfacial_area(volume, spacing=spacing)
    vpore = pore_volume(volume, spacing=spacing)
    return 0.0 if vpore <= 0.0 else float(area / vpore)


def specific_surface_area_summary(volume: np.ndarray, spacing: Optional[Union[float, Tuple[float, float, float]]] = None):
    counts = interface_area_counts(volume)
    area = interfacial_area(volume, spacing=spacing)
    vbulk = bulk_volume(volume, spacing=spacing)
    vpore = pore_volume(volume, spacing=spacing)
    return {
        "interface_faces_x": counts["faces_x"],
        "interface_faces_y": counts["faces_y"],
        "interface_faces_z": counts["faces_z"],
        "interface_faces_total": counts["faces_total"],
        "interfacial_area": area,
        "bulk_volume": vbulk,
        "pore_volume": vpore,
        "ssa_bulk": specific_surface_area_bulk(volume, spacing=spacing),
        "ssa_pore": specific_surface_area_pore(volume, spacing=spacing),
    }


def specific_surface_area_profile(volume: np.ndarray, axis: int = 2, spacing: Optional[Union[float, Tuple[float, float, float]]] = None) -> np.ndarray:
    dx, dy, dz = voxel_spacing_tuple(spacing)
    if axis == 0:
        n = volume.shape[0] - 1
        out = np.zeros(n, dtype=float)
        for i in range(n):
            out[i] = specific_surface_area_bulk(volume[i:i+2, :, :], spacing=(dx, dy, dz))
        return out
    if axis == 1:
        n = volume.shape[1] - 1
        out = np.zeros(n, dtype=float)
        for j in range(n):
            out[j] = specific_surface_area_bulk(volume[:, j:j+2, :], spacing=(dx, dy, dz))
        return out
    if axis == 2:
        n = volume.shape[2] - 1
        out = np.zeros(n, dtype=float)
        for k in range(n):
            out[k] = specific_surface_area_bulk(volume[:, :, k:k+2], spacing=(dx, dy, dz))
        return out
    raise ValueError("axis must be 0, 1, or 2")


def plot_specific_surface_area_profile(volume, axis=2, spacing=None):
    prof = specific_surface_area_profile(volume, axis=axis, spacing=spacing)
    depth = np.arange(len(prof))
    axis_name = {0: "x", 1: "y", 2: "z"}[axis]
    plt.figure(figsize=(6.5, 4))
    plt.plot(depth, prof, marker="o", ms=3)
    plt.xlabel(f"Slab index along {axis_name}")
    plt.ylabel("Specific surface area (A / V_bulk)")
    plt.title(f"Specific surface area profile along {axis_name}")
    plt.tight_layout()
    plt.show()

In [ ]:

def plot_grayscale_slices(volume: np.ndarray, title: str = "Imported raw volume") -> None:
    nx, ny, nz = volume.shape
    cx, cy, cz = nx // 2, ny // 2, nz // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    im0 = axes[0].imshow(volume[:, :, cz].T, origin="lower", cmap="gray")
    axes[0].set_title(f"XY slice at z={cz}")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")
    im1 = axes[1].imshow(volume[:, cy, :].T, origin="lower", cmap="gray")
    axes[1].set_title(f"XZ slice at y={cy}")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("z")
    im2 = axes[2].imshow(volume[cx, :, :].T, origin="lower", cmap="gray")
    axes[2].set_title(f"YZ slice at x={cx}")
    axes[2].set_xlabel("y")
    axes[2].set_ylabel("z")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_histogram(volume: np.ndarray, bins: int = 256) -> None:
    plt.figure(figsize=(6.5, 4))
    plt.hist(volume.ravel(), bins=bins)
    plt.xlabel("Voxel intensity")
    plt.ylabel("Count")
    plt.title("Volume intensity histogram")
    plt.tight_layout()
    plt.show()


BENTHEIMER_METADATA = {
    "Benth_4.54.raw": {"shape": (575, 575, 575), "voxel_size_microns": 4.54},
    "Benth_6.28.raw": {"shape": (415, 415, 415), "voxel_size_microns": 6.28},
    "Benth_8.33.raw": {"shape": (313, 313, 313), "voxel_size_microns": 8.33},
    "Benth_10.23.raw": {"shape": (255, 255, 255), "voxel_size_microns": 10.23},
}


def load_raw_volume(filename: str, shape: Tuple[int, int, int], dtype: np.dtype = np.dtype("<u1")) -> np.ndarray:
    expected = int(np.prod(shape))
    expected_bytes = expected * dtype.itemsize
    actual_bytes = os.path.getsize(filename)
    if actual_bytes != expected_bytes:
        raise ValueError(f"File size mismatch: expected {expected_bytes} bytes for shape {shape}, found {actual_bytes} bytes.")
    data = np.fromfile(filename, dtype=dtype)
    return data.reshape(shape)


def binarize_segmented_volume(volume: np.ndarray, threshold: int = 127) -> np.ndarray:
    return (volume > threshold).astype(np.uint8)


def crop_central_subvolume(volume: np.ndarray, crop_shape: Tuple[int, int, int]) -> np.ndarray:
    nx, ny, nz = volume.shape
    cx, cy, cz = nx // 2, ny // 2, nz // 2
    sx, sy, sz = crop_shape
    if sx > nx or sy > ny or sz > nz:
        raise ValueError("crop_shape must be smaller than or equal to the original volume shape")
    x0 = cx - sx // 2
    y0 = cy - sy // 2
    z0 = cz - sz // 2
    return volume[x0:x0 + sx, y0:y0 + sy, z0:z0 + sz]


def downsample_binary_volume(volume: np.ndarray, factor: int = 2) -> np.ndarray:
    if factor < 1:
        raise ValueError("factor must be >= 1")
    if factor == 1:
        return volume.copy()
    nx, ny, nz = volume.shape
    nx2 = (nx // factor) * factor
    ny2 = (ny // factor) * factor
    nz2 = (nz // factor) * factor
    trimmed = volume[:nx2, :ny2, :nz2]
    reshaped = trimmed.reshape(nx2 // factor, factor, ny2 // factor, factor, nz2 // factor, factor)
    block_mean = reshaped.mean(axis=(1, 3, 5))
    return (block_mean > 0.5).astype(np.uint8)

In [ ]:

def safe_round(value, ndigits: int = 4):
    if value is None:
        return None
    if isinstance(value, str):
        return value
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return round(float(value), ndigits)
    return value


def summarize_volume(volume: np.ndarray, name: str = "sample", spacing=None, radial_bin_width: float = 1.0):
    summary = {
        "name": name,
        "shape": tuple(int(x) for x in volume.shape),
        "porosity": measured_porosity(volume),
        "interface_fraction": interface_fraction(volume),
    }
    summary.update(connected_pore_statistics(volume))
    summary.update(correlation_length_summary(volume))
    summary.update(tortuosity_summary(volume))
    summary.update(radial_correlation_summary(volume, spacing=spacing, bin_width=radial_bin_width))
    summary.update(specific_surface_area_summary(volume, spacing=spacing))
    return summary


def print_summary_table(summaries):
    if not summaries:
        print("No summaries to display.")
        return
    keys = []
    for s in summaries:
        for k in s.keys():
            if k not in keys:
                keys.append(k)
    metric_width = max(len(str(k)) for k in keys) + 2
    name_widths = [max(len(str(s.get("name", "sample"))), 12) for s in summaries]
    header = "Metric".ljust(metric_width)
    for s, w in zip(summaries, name_widths):
        header += str(s.get("name", "sample")).ljust(w + 2)
    print(header)
    print("-" * len(header))
    for key in keys:
        row = str(key).ljust(metric_width)
        for s, w in zip(summaries, name_widths):
            value = safe_round(s.get(key, None))
            row += str(value).ljust(w + 2)
        print(row)


def compare_two_volumes_slices(volume_a, volume_b, name_a="A", name_b="B"):
    cmap = porous_cmap()
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for row, (vol, name) in enumerate([(volume_a, name_a), (volume_b, name_b)]):
        nx, ny, nz = vol.shape
        cx, cy, cz = nx // 2, ny // 2, nz // 2
        axes[row, 0].imshow(vol[:, :, cz].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
        axes[row, 0].set_title(f"{name}: XY at z={cz}")
        axes[row, 1].imshow(vol[:, cy, :].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
        axes[row, 1].set_title(f"{name}: XZ at y={cy}")
        axes[row, 2].imshow(vol[cx, :, :].T, origin="lower", cmap=cmap, vmin=0, vmax=1)
        axes[row, 2].set_title(f"{name}: YZ at x={cx}")
    plt.tight_layout()
    plt.show()


def to_cfd_convention(volume: np.ndarray, pore_value: int = 1, solid_value: int = 0) -> np.ndarray:
    out = np.empty_like(volume, dtype=np.uint8)
    out[volume == 0] = pore_value
    out[volume == 1] = solid_value
    return out


def export_numpy_volume(volume: np.ndarray, filename: str) -> None:
    np.save(filename, volume.astype(np.uint8))
    print(f"Saved NumPy volume: {filename}")


def export_raw_volume(volume: np.ndarray, filename: str) -> None:
    volume.astype(np.uint8).tofile(filename)
    print(f"Saved raw volume: {filename}")


def export_metadata_json(filename: str, volume: np.ndarray, voxel_size_microns: Optional[float] = None, convention: str = "0=pore, 1=solid", source_name: str = "sample") -> None:
    metadata = {
        "source_name": source_name,
        "shape": [int(x) for x in volume.shape],
        "dtype": "uint8",
        "convention": convention,
        "voxel_size_microns": voxel_size_microns,
    }
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"Saved metadata JSON: {filename}")


def export_mhd_header(header_filename: str, raw_filename: str, shape: Tuple[int, int, int], voxel_size_microns: Optional[float] = None, element_type: str = "MET_UCHAR") -> None:
    if voxel_size_microns is None:
        spacing = (1.0, 1.0, 1.0)
    else:
        spacing = (float(voxel_size_microns), float(voxel_size_microns), float(voxel_size_microns))
    header_lines = [
        "ObjectType = Image",
        "NDims = 3",
        "BinaryData = True",
        "BinaryDataByteOrderMSB = False",
        "CompressedData = False",
        "TransformMatrix = 1 0 0 0 1 0 0 0 1",
        "Offset = 0 0 0",
        "CenterOfRotation = 0 0 0",
        f"ElementSpacing = {spacing[0]} {spacing[1]} {spacing[2]}",
        f"DimSize = {shape[0]} {shape[1]} {shape[2]}",
        "ElementNumberOfChannels = 1",
        f"ElementType = {element_type}",
        f"ElementDataFile = {Path(raw_filename).name}",
    ]
    header_text = "\n".join(header_lines) + "\n"
    with open(header_filename, "w", encoding="utf-8") as f:
        f.write(header_text)
    print(f"Saved MHD header: {header_filename}")


def export_volume_bundle(volume: np.ndarray, base_name: str, out_dir: str = "exports", voxel_size_microns: Optional[float] = None, source_name: str = "sample", convert_to_cfd: bool = True):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    if convert_to_cfd:
        volume_to_save = to_cfd_convention(volume, pore_value=1, solid_value=0)
        convention = "1=pore(fluid), 0=solid"
    else:
        volume_to_save = volume.astype(np.uint8)
        convention = "0=pore, 1=solid"
    npy_file = out_path / f"{base_name}.npy"
    raw_file = out_path / f"{base_name}.raw"
    json_file = out_path / f"{base_name}.json"
    mhd_file = out_path / f"{base_name}.mhd"
    export_numpy_volume(volume_to_save, str(npy_file))
    export_raw_volume(volume_to_save, str(raw_file))
    export_metadata_json(str(json_file), volume_to_save, voxel_size_microns=voxel_size_microns, convention=convention, source_name=source_name)
    export_mhd_header(str(mhd_file), str(raw_file), shape=volume_to_save.shape, voxel_size_microns=voxel_size_microns)
    return {"npy": str(npy_file), "raw": str(raw_file), "json": str(json_file), "mhd": str(mhd_file)}

In [ ]:

lattice, history = run_simulation(cfg)

plot_history(history)
plot_slices(lattice, title="Synthetic porous morphology")

print("Porosity:", measured_porosity(lattice))
print("Connectivity:", connected_pore_statistics(lattice))
print("Tortuosity proxy:", tortuosity_summary(lattice))
print("1D correlation summary:", correlation_length_summary(lattice))
print("3D radial correlation summary:", radial_correlation_summary(lattice, spacing=1.0, bin_width=1.0))
print("SSA summary:", specific_surface_area_summary(lattice, spacing=1.0))

lags_x, corr_x = one_dimensional_correlation(lattice, axis=0)
plot_correlation(lags_x, corr_x, axis_name="x")
plot_correlations_xyz(lattice)

labels_syn, sizes_syn = label_phase_clusters(lattice, phase=0)
largest_syn = largest_cluster_mask(labels_syn, sizes_syn)
plot_boolean_slices(largest_syn, title="Largest connected pore cluster")

plot_depth_profiles(lattice, axis=2)
plot_slice_connectivity_profiles(lattice, axis=2)

print("")
print("Transport-oriented pore-network summary")
print("-" * 40)
for key, value in tortuosity_summary(lattice).items():
    print(f"{key}: {value}")

plot_breakthrough_profile(lattice, axis=2)
plot_distance_field_slices(lattice, axis=2)

plot_full_3d_radial_correlation(
    lattice,
    spacing=1.0,
    bin_width=1.0,
    title="Synthetic sample: full 3D radial autocorrelation",
    label="synthetic",
)

plot_specific_surface_area_profile(lattice, axis=2, spacing=1.0)

In [ ]:

raw_filename = "Benth_10.23.raw"
raw_dtype = np.dtype("<u1")

if raw_filename not in BENTHEIMER_METADATA:
    raise ValueError(f"Unknown Bentheimer filename: {raw_filename}")

raw_shape = BENTHEIMER_METADATA[raw_filename]["shape"]
voxel_size_microns = BENTHEIMER_METADATA[raw_filename]["voxel_size_microns"]

print("Selected file:", raw_filename)
print("Shape:", raw_shape)
print("Voxel size (microns):", voxel_size_microns)

if not os.path.exists(raw_filename):
    print("")
    print("The file was not found in the current working directory.")
    print("Put the .raw file next to this notebook or change raw_filename to the correct path.")
else:
    vol_raw = load_raw_volume(raw_filename, raw_shape, raw_dtype)

    print("")
    print("Loaded volume")
    print("shape:", vol_raw.shape)
    print("dtype:", vol_raw.dtype)
    print("min/max:", vol_raw.min(), vol_raw.max())
    print("unique values:", np.unique(vol_raw))

    plot_grayscale_slices(vol_raw, title="Bentheimer raw volume")
    plot_histogram(vol_raw)

    lattice_ct = binarize_segmented_volume(vol_raw, threshold=127)
    analysis_volume = lattice_ct

    print("")
    print("Binary imported lattice")
    print("porosity:", measured_porosity(analysis_volume))
    print("unique values:", np.unique(analysis_volume))

    plot_slices(analysis_volume, title="Imported Bentheimer lattice: 0=pore, 1=solid")

    print("")
    print("Imported-volume analysis")
    print("-" * 40)
    print("Porosity:", measured_porosity(analysis_volume))
    print("Interface fraction:", interface_fraction(analysis_volume))
    print("Connectivity:", connected_pore_statistics(analysis_volume))
    print("Tortuosity proxy:", tortuosity_summary(analysis_volume))
    print("1D correlation summary:", correlation_length_summary(analysis_volume))
    print(
        "3D radial correlation summary:",
        radial_correlation_summary(
            analysis_volume,
            spacing=voxel_size_microns,
            bin_width=voxel_size_microns,
        ),
    )
    print(
        "SSA summary:",
        specific_surface_area_summary(
            analysis_volume,
            spacing=voxel_size_microns,
        ),
    )

    plot_correlations_xyz(analysis_volume)

    labels_ct, sizes_ct = label_phase_clusters(analysis_volume, phase=0)
    largest_ct = largest_cluster_mask(labels_ct, sizes_ct)
    plot_boolean_slices(largest_ct, title="Imported CT: largest connected pore cluster")

    plot_depth_profiles(analysis_volume, axis=2)
    plot_slice_connectivity_profiles(analysis_volume, axis=2)

    print("")
    print("Transport-oriented pore-network summary")
    print("-" * 40)
    for key, value in tortuosity_summary(analysis_volume).items():
        print(f"{key}: {value}")

    plot_breakthrough_profile(analysis_volume, axis=2)
    plot_distance_field_slices(analysis_volume, axis=2)

    plot_full_3d_radial_correlation(
        analysis_volume,
        spacing=voxel_size_microns,
        bin_width=voxel_size_microns,
        title="Imported CT: full 3D radial autocorrelation",
        label="imported_CT",
    )

    plot_specific_surface_area_profile(
        analysis_volume,
        axis=2,
        spacing=voxel_size_microns,
    )

In [ ]:

if "analysis_volume" not in globals():
    print("analysis_volume is not defined yet. Run the CT import cell first.")
else:
    synthetic_summary = summarize_volume(
        lattice,
        name="synthetic",
        spacing=1.0,
        radial_bin_width=1.0,
    )

    imported_summary = summarize_volume(
        analysis_volume,
        name="imported_CT",
        spacing=voxel_size_microns,
        radial_bin_width=voxel_size_microns,
    )

    print_summary_table([synthetic_summary, imported_summary])

    compare_two_volumes_slices(
        lattice,
        analysis_volume,
        name_a="synthetic",
        name_b="imported_CT",
    )

    compare_full_3d_radial_correlations(
        lattice,
        analysis_volume,
        spacing_a=1.0,
        spacing_b=voxel_size_microns,
        bin_width_a=1.0,
        bin_width_b=voxel_size_microns,
        label_a="synthetic",
        label_b="imported_CT",
        title="Full 3D radial autocorrelation: synthetic vs imported CT",
    )

    exported_ct = export_volume_bundle(
        analysis_volume,
        base_name="bentheimer_ct_cfd",
        out_dir="exports",
        voxel_size_microns=voxel_size_microns,
        source_name="Bentheimer imported CT",
        convert_to_cfd=True,
    )

    exported_syn = export_volume_bundle(
        lattice,
        base_name="synthetic_porosity_cfd",
        out_dir="exports",
        voxel_size_microns=None,
        source_name="Synthetic porous sample",
        convert_to_cfd=True,
    )

    print("")
    print("Exported CT files:")
    print(exported_ct)
    print("")
    print("Exported synthetic files:")
    print(exported_syn)